# Hetionet: original DREAMwalk, local run

Run the authors' original DREAMwalk on **Hetionet v1.0** and compare with **Figure 2c**. This tests the pipeline on a fixed public molecular graph with native Disease Ontology IDs.

**Select any local Python 3 kernel and run the notebook from top to bottom.** Computation uses the separately installed DREAMwalk Python 3.8 environment. Training starts when you run the computation cells. Published notebooks have cleared outputs.

The **original teleport selector and original C++ `pp=1`** are retained. MSI diagnostic patches are not installed. Accepted settings remain fixed; this is not a parameter search.

Inputs, embeddings, logs and results use the separate `hetionet_seed0_original` run directories.


## 1. Paths, parameters and paper reference

Open from the repository root, model directory or `notebooks/`. The notebook imports lightweight I/O helpers; scientific computation runs in a Python 3.8 subprocess. Add future Hetionet controls as sections in this notebook.


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import JSON, display

sys.dont_write_bytecode = True
candidates = []
for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    candidates.extend((base, base / "00.Reproduction" / "00.DREAMwalk_reproduction"))
ROOT = next((base for base in candidates
             if (base / "src" / "run_hetionet_original.py").is_file()
             and (base / "vendor" / "DREAMwalk_upstream").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("DREAMwalk project not found; open the notebook from the repository or model directory.")

sys.path.insert(0, str(ROOT / "src"))
import dreamwalk_notebook_workflow as workflow
import hetionet_inputs

PYTHON = ROOT / "environment" / ".conda-env" / "bin" / "python"
RUNNER = ROOT / "src" / "run_hetionet_original.py"
RUN_NAME = "hetionet_seed0_original"
DATA = ROOT / "data" / "embeddings" / RUN_NAME
RESULTS = ROOT / "results" / RUN_NAME
LOGS = ROOT / "logs" / RUN_NAME
INPUTS = hetionet_inputs.paths(ROOT)
EMBEDDING = DATA / "hetionet_seed0_original.pkl"

SETTINGS = dict(tp_factor=0.3, cutoff=0.4, seed=0, num_walks=100,
                walk_length=10, dimension=128, window_size=4, workers=4,
                em_max_iter=5, p=1, q=1, pp=1, epochs=1, negative=5,
                sample=0.001, folds=10, cv_seed=0)
actual = workflow.runner_constants(RUNNER, "SETTINGS")["SETTINGS"]
workflow.check_parameters(actual, SETTINGS)
if actual != SETTINGS:
    raise ValueError("Notebook and runner parameter dictionaries differ.")
if not PYTHON.is_file():
    raise FileNotFoundError("Python 3.8 interpreter not found: " + str(PYTHON))

print("Project: ", ROOT)
print("Computation interpreter: ", PYTHON)
print("Run name：", RUN_NAME)
print("Embedding：", EMBEDDING)
print("Logs：", LOGS)
workflow.show_rows([dict(parameter=key, value=value) for key, value in SETTINGS.items()])

Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction
Computation interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python
Run name： hetionet_seed0_original
Embedding： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/hetionet_seed0_original.pkl
Logs： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/hetionet_seed0_original


parameter,value
tp_factor,0.300000
cutoff,0.400000
seed,0
num_walks,100
walk_length,10
dimension,128
window_size,4
workers,4
em_max_iter,5
p,1


### Fixed training and evaluation settings

- Molecular graph: the ten molecular relation types in Supplementary Table 3; `CtD` treatment labels remain separate.
- Similarity: upstream IC/similarity code, cutoff `0.4`; teleport factor `τ=0.3`.
- Walks: `100 × active nodes`, length `10`, at most `5` edge-type EM iterations, `p=q=1`, `4` workers.
- Heterogeneous Skip-gram: original C++ `pp=1`, 128 dimensions, window parameter `4`, one epoch, five negatives, subsampling `0.001`.
- Features: `drug embedding − disease embedding`; released XGBoost settings, 500 trees and depth 6.
- Evaluation: **one seed-0 negative set × ten folds**. Figure 2c contains ten experimental repeats. This notebook does not automatically expand to 100 classifier fits.
- No extra startup `PYTHONHASHSEED` is set. Upstream `set_seed(0)` remains unchanged; worker RNG states and four-thread training are not asserted to be deterministic.

References are extracted from **Source Data / Figure2c / B2:D11**, not estimated from the figure. Provenance records the workbook hash, cells, ten raw rows and seeds.


In [2]:
REFERENCE = ROOT / "references" / "hetionet_fig2c_dreamwalk.csv"
REFERENCE_PROVENANCE = ROOT / "references" / "hetionet_fig2c_dreamwalk_provenance.json"
workflow.show_csv(REFERENCE)
display(JSON(workflow.read_json(REFERENCE_PROVENANCE), expanded=False))

hetionet_fig2c_dreamwalk.csv


metric,mean
accuracy,0.852055
auroc,0.918076
aupr,0.915982


<IPython.core.display.JSON object>

## 2. Check the environment and upstream commit

Check Python 3.8, package pins, commit `2f93723c3fbfa899d0f23c02e3459de8bfac0581` and unchanged tracked upstream files. Restore upstream Git metadata as described in the repository README before this step.


In [3]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "environment", RUN_NAME)

Running: run_hetionet_original.py environment | hetionet_seed0_original
{
  "packages": {
    "numpy": "1.23.3",
    "scipy": "1.9.1",
    "scikit-learn": "1.1.3",
    "networkx": "2.8.8",
    "pandas": "1.4.4",
    "tqdm": "4.64.1",
    "parmap": "1.6.0",
    "xgboost": "1.7.4"
  },
  "upstream_commit": "2f93723c3fbfa899d0f23c02e3459de8bfac0581",
  "settings": {
    "tp_factor": 0.3,
    "cutoff": 0.4,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "dimension": 128,
    "window_size": 4,
    "workers": 4,
    "em_max_iter": 5,
    "p": 1,
    "q": 1,
    "pp": 1,
    "epochs": 1,
    "negative": 5,
    "sample": 0.001,
    "folds": 10,
    "cv_seed": 0
  },
  "xgboost": {
    "base_score": 0.5,
    "booster": "gbtree",
    "eval_metric": "error",
    "objective": "binary:logistic",
    "gamma": 0,
    "learning_rate": 0.1,
    "max_depth": 6,
    "n_estimators": 500,
    "tree_method": "auto",
    "min_child_weight": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.

## 3. Download fixed public inputs

Download four pinned files: Hetionet nodes, Hetionet edges, historical HumanDO OBO and the corresponding DrugBank slim. Each has a fixed URL, size and SHA-256. Reuse existing matching files.

DrugBank IDs and DOIDs connect directly to the hierarchies. Disease mapping does not use MSI's UMLS-to-MeSH route.


In [4]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "download", RUN_NAME)

Running: run_hetionet_original.py download | hetionet_seed0_original


In [5]:
workflow.show_rows([
    dict(file=source.name, bytes=source.size, sha256=source.sha256,
         saved=(INPUTS["raw_dir"] / source.name).is_file(), url=source.url)
    for source in hetionet_inputs.SOURCES
])

file,bytes,sha256,saved,url
hetionet-v1.0-nodes.tsv,2509984,3ad9eb86c031df8bc7d5efefa6edc93562182eb9a6a71e5129d3f18b9d42c50e,True,https://raw.githubusercontent.com/hetio/hetionet/4933ca17639cdcf146f91cc44e28128cbfc6fd24/hetnet/tsv/hetionet-v1.0-nodes.tsv
hetionet-v1.0-edges.sif.gz,12401407,611f0411ac666be4e0270e54bbca67407e14720bfc3099311f0f5eec25c5a947,True,https://raw.githubusercontent.com/hetio/hetionet/4933ca17639cdcf146f91cc44e28128cbfc6fd24/hetnet/tsv/hetionet-v1.0-edges.sif.gz
HumanDO.obo,4014997,220921d830b1b8c71054e5e3fc73f0774ce35e2406f4e33eda18021a50535364,True,https://raw.githubusercontent.com/dhimmel/disease-ontology/75050ea2d4f60e745d3f3578ae03560a2cc0e444/download/HumanDO.obo
drugbank-slim.tsv,441087,cfb847f197862f6642484d6bb4ed96473d23235873a0ee4d4a9ebd8dc63d3613,True,https://raw.githubusercontent.com/dhimmel/drugbank/3e87872db5fca5ac427ce27464ab945c0ceb4ec6/data/drugbank-slim.tsv


## 4. Build molecular and hierarchy inputs; save pairs and folds

### Relation to Supplementary Table 3

| Item | Raw four-type nodes/labels | Active graph/evaluable labels |
|---|---:|---:|
| Nodes | 24,456 | **21,896** |
| Compounds | 1,552 | 1,440 |
| Diseases | 137 | 134 |
| Genes | 20,945 | 18,500 |
| Pathways | 1,822 | 1,822 |
| CtD positives | 755 | **751** |

The ten molecular relation types contain **638,304 raw rows**. Upstream `read_graph` uses `MultiDiGraph.to_undirected()`, producing **635,741 MultiGraph edges**. The **634,482** unique unordered endpoint pairs are a different count.

Keep original row order and the original reader. Do not add self-loops or treatment edges. After removing molecular isolates, four CtD labels lack a drug endpoint. Record these in `excluded_positive_pairs.csv`.

### Semantic inputs and label scope

- Historical DrugBank slim ATC coverage: **1,188 / 1,440 active drugs**.
- Native Disease Ontology coverage: **134 / 134 active diseases**. Audit explicit `alt_id` resolution.
- Genes use type `gene`; pathways use `etc`, following upstream `d/i/g/e` prefixes.
- The historical hierarchy can be rebuilt from pinned sources; it is not claimed to be the authors' exact snapshot.
- CtD defines positive labels. CpD is a different relation and is not added. Negative candidates are active drug–disease pairs excluding all raw CtD positives.
- Sample negatives once with `random.Random(0)` and save them. The run configuration reports any sampled CpD pairs.
- Save the order of 751 positives plus 751 negatives and all ten test-fold IDs.

**This uses the original algorithm with explicit reconstruction rules; it does not recover the authors' exact 755-pair benchmark.**


In [6]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "prepare", RUN_NAME)

Running: run_hetionet_original.py prepare | hetionet_seed0_original
Prepared inputs and saved negative/fold IDs reused


In [7]:
if INPUTS["manifest"].is_file():
    manifest = workflow.read_json(INPUTS["manifest"])
    display(JSON(manifest, expanded=False))
    workflow.show_rows(manifest["molecular"]["edge_types"])
    print("Excluded positive pairs：")
    workflow.show_csv(INPUTS["audit_dir"] / "excluded_positive_pairs.csv")
    print("Full audits: ", INPUTS["audit_dir"])
else:
    print("Inputs are not prepared; run the prepare cell above first.")

config_path = RESULTS / "run_config.json"
if config_path.is_file():
    config = workflow.read_json(config_path)
    display(JSON(dict(counts=config["counts"],
                      negative_policy=config["negative_policy"],
                      notes=config["notes"]), expanded=True))
    print("Negative pairs：", RESULTS / "negative_pairs.csv")
    print("Fold IDs：", RESULTS / "fold_assignments.csv")
else:
    print("No saved run configuration.")

<IPython.core.display.JSON object>

id,metaedge,rows
1,CbG,11571
2,CuG,18756
3,CdG,21102
4,DaG,12623
5,DuG,7731
6,DdG,7623
7,Gr>G,265672
8,GiG,147164
9,GcG,61690
10,GpPW,84372


Excluded positive pairs：
excluded_positive_pairs.csv


drug,disease,absent_endpoints
Compound::DB00888,Disease::DOID:2531,Compound::DB00888
Compound::DB00258,Disease::DOID:11476,Compound::DB00258
Compound::DB00888,Disease::DOID:0060073,Compound::DB00888
Compound::DB01265,Disease::DOID:2043,Compound::DB01265


Full audits: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/hetionet_inputs


<IPython.core.display.JSON object>

Negative pairs： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/hetionet_seed0_original/negative_pairs.csv
Fold IDs： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/hetionet_seed0_original/fold_assignments.csv


## 5. Generate semantic similarity with upstream code

Call upstream `save_sim_graph` on the DO and ATC hierarchies at cutoff `0.4`. This generates the similarity graph without training embeddings.


In [8]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "similarity", RUN_NAME)

Running: run_hetionet_original.py similarity | hetionet_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/similarity_graph.tsv


In [9]:
marker = RESULTS / "similarity_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=True))
else:
    print("Similarity graph is not complete.")

<IPython.core.display.JSON object>

## 6. Original EM → full walks → C++ `pp=1` embeddings

Call unchanged upstream `save_embedding_files`. Retain its EM, traversal, teleport selector and heterogeneous Skip-gram.

Expect **2,189,600 walks** of length `10`. This is the main computation stage; output is also saved to this run's `embedding.log`.

**Atomically save the completed `.pkl` and its completion marker before starting XGBoost.** An interrupted evaluation can reuse this embedding. Upstream combines EM, walk generation and C++ training in one call; incomplete embeddings are not accepted as checkpoints.


In [10]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "embedding", RUN_NAME)

Running: run_hetionet_original.py embedding | hetionet_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/similarity_graph.tsv
random seed with 0
Reading network files...
Training edge type transition matrix...
Generating paths...
Generating node embeddings...
DREAMwalk/HeterogeneousSG.cpp:328:16: warning: variable 'i' set but not used [-Wunused-but-set-variable]
  328 |   long long a, i = 0;
      |                ^
1 warning generated.
Node embeddings saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/hetionet_seed0_original/original_embedding_nic32ntj/embedding_pending.pkl
EMBEDDING SAVED before evaluation: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/hetionet_seed0_original.pkl


In [11]:
marker = RESULTS / "embedding_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=True))
    print("Saved embedding: ", EMBEDDING)
else:
    print("Embedding is not complete. Destination: ", EMBEDDING)

<IPython.core.display.JSON object>

Saved embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/hetionet_seed0_original.pkl


## 7. XGBoost evaluation on saved pairs and folds

Read the saved embedding and preparation-stage pair/fold IDs. Save metrics after each fold and reuse completed folds on restart.

Accuracy uses `prob.round()`, AUROC uses predicted probabilities, and AUPR uses `average_precision_score`. Run ten folds for this seed-0 set only.


In [12]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "evaluate", RUN_NAME)

Running: run_hetionet_original.py evaluate | hetionet_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/hetionet_seed0_original/hetionet_seed0_original.pkl
Fold 1 {'accuracy': 0.834437, 'auroc': 0.912807, 'aupr': 0.910254}
Fold 2 {'accuracy': 0.907285, 'auroc': 0.952807, 'aupr': 0.95219}
Fold 3 {'accuracy': 0.86, 'auroc': 0.9328, 'aupr': 0.935135}
Fold 4 {'accuracy': 0.826667, 'auroc': 0.914489, 'aupr': 0.9121}
Fold 5 {'accuracy': 0.8, 'auroc': 0.901689, 'aupr': 0.908133}
Fold 6 {'accuracy': 0.806667, 'auroc': 0.908267, 'aupr': 0.904923}
Fold 7 {'accuracy': 0.86, 'auroc': 0.9248, 'aupr': 0.887214}
Fold 8 {'accuracy': 0.866667, 'auroc': 0.940267, 'aupr': 0.927621}
Fold 9 {'accuracy': 0.873333, 'auroc': 0.950044, 'aupr': 0.952639}
Fold 10 {'accuracy': 0.86, 'auroc': 0.930844, 'aupr': 0.928462}
Ten-fold evaluation saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction

In [13]:
workflow.show_csv(RESULTS / "fold_results.csv")
workflow.show_csv(RESULTS / "summary.csv")

fold_results.csv


fold,accuracy,auroc,aupr,n_train,n_test,seconds
1,0.834437,0.912807,0.910254,1351,151,1.443363
2,0.907285,0.952807,0.952190,1351,151,1.468635
3,0.860000,0.932800,0.935135,1352,150,1.435635
4,0.826667,0.914489,0.912100,1352,150,1.425910
5,0.800000,0.901689,0.908133,1352,150,1.427774
6,0.806667,0.908267,0.904923,1352,150,1.436064
7,0.860000,0.924800,0.887214,1352,150,1.425187
8,0.866667,0.940267,0.927621,1352,150,1.467091
9,0.873333,0.950044,0.952639,1352,150,1.440170
10,0.860000,0.930844,0.928462,1352,150,1.435917


summary.csv


metric,mean,SD,n
accuracy,0.849506,0.032602,10
auroc,0.926881,0.017562,10
aupr,0.921867,0.021150,10


## 8. Compare with Figure 2c

Report `local − paper` for all three metrics, then assess whether the MSI discrepancy appears dataset-specific.

The paper reference averages ten Source Data rows; the local result averages ten folds for one negative set. Report the four excluded CtD labels and historical semantic-source differences alongside performance.

Stop after this section. No additional seeds, parameter search or MSI controls are started.


In [14]:
workflow.run_phase(ROOT, PYTHON, "run_hetionet_original.py", "compare", RUN_NAME)

Running: run_hetionet_original.py compare | hetionet_seed0_original
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/hetionet_seed0_original/summary.csv
[
  {
    "metric": "accuracy",
    "local_seed0_10fold": 0.8495055187637969,
    "paper_10repeat_mean": 0.8520547945205478,
    "local_minus_paper": -0.0025492757567509017
  },
  {
    "metric": "auroc",
    "local_seed0_10fold": 0.926881403508772,
    "paper_10repeat_mean": 0.9180756239444549,
    "local_minus_paper": 0.008805779564317162
  },
  {
    "metric": "aupr",
    "local_seed0_10fold": 0.9218670646914494,
    "paper_10repeat_mean": 0.9159818403271082,
    "local_minus_paper": 0.005885224364341246
  }
]


In [15]:
workflow.show_csv(RESULTS / "paper_comparison.csv")
marker = RESULTS / "comparison_complete.json"
if marker.is_file():
    display(JSON(workflow.read_json(marker), expanded=False))
else:
    print("No final comparison; run the preceding cell after evaluation completes.")
print("Results directory: ", RESULTS)
print("Logs directory: ", LOGS)

paper_comparison.csv


metric,local_seed0_10fold,paper_10repeat_mean,local_minus_paper
accuracy,0.849506,0.852055,-0.002549
auroc,0.926881,0.918076,0.008806
aupr,0.921867,0.915982,0.005885


<IPython.core.display.JSON object>

Results directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/hetionet_seed0_original
Log directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/logs/hetionet_seed0_original


## Sources

- [DREAMwalk paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC10272215/)
- [Author Source Data](https://pmc.ncbi.nlm.nih.gov/articles/instance/10272215/bin/41467_2023_39301_MOESM4_ESM.xlsx), `Figure2c`, rows 2–11.
- [Pinned DREAMwalk](https://github.com/eugenebang/DREAMwalk/tree/2f93723c3fbfa899d0f23c02e3459de8bfac0581)
- [Hetionet v1.0.0](https://github.com/hetio/hetionet/tree/4933ca17639cdcf146f91cc44e28128cbfc6fd24), fixed node/edge source corresponding to Supplementary Table 3.
- [Historical Disease Ontology](https://github.com/dhimmel/disease-ontology/tree/75050ea2d4f60e745d3f3578ae03560a2cc0e444)
- [Historical DrugBank slim](https://github.com/dhimmel/drugbank/tree/3e87872db5fca5ac427ce27464ab945c0ceb4ec6)

Runtime URLs, hashes and audits are saved in `results/hetionet_inputs/manifest.json`. Paper-reference extraction is recorded in `references/hetionet_fig2c_dreamwalk_provenance.json`.
